# Querying data.gov.cz
* trying GraphQL and SPARQL

## Using graphql
GraphQL is not as powerful as SPARQL.

In [4]:
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

transport = RequestsHTTPTransport(
    url="https://data.gov.cz/graphql",
    verify=True,
    retries=3,
)

client = Client(transport=transport, fetch_schema_from_transport=False)

query = gql("""
{
  datasets(limit: 10) {
    data {
      iri
      title {
        cs
      }
      description {
        cs
      }
      publisher {
        title {
          cs
        }
      }
    }
  }
}
""")
result = client.execute(query)
result

{'datasets': {'data': [{'iri': 'https://data.gov.cz/zdroj/datové-sady/00006947/b0cb0395d7d9c86f4fba94f2de9132d3',
    'title': {'cs': 'Příloha - období 03/2020'},
    'description': {'cs': 'Účetní výkazy: Příloha'},
    'publisher': {'title': {'cs': 'Ministerstvo financí'}}},
   {'iri': 'https://data.gov.cz/zdroj/datové-sady/00006947/247a76a4aeaaa197840ef8ae32db79f5',
    'title': {'cs': 'Příloha'},
    'description': {'cs': 'Účetní výkazy: Příloha'},
    'publisher': {'title': {'cs': 'Ministerstvo financí'}}},
   {'iri': 'https://data.gov.cz/zdroj/datové-sady/00006947/a5797873989d639bb8972c54e8789e48',
    'title': {'cs': 'FIN 1-12 SF - Plnění rozpočtu SF - období 09/2016'},
    'description': {'cs': 'Finanční výkaz pro hodnocení plnění rozpočtu státních fondů. Výkaz byl zaveden od roku 2015.'},
    'publisher': {'title': {'cs': 'Ministerstvo financí'}}},
   {'iri': 'https://data.gov.cz/zdroj/datové-sady/00006947/a3fcaae1acd521636237ba9335b8b679',
    'title': {'cs': 'FIN 1-12 SF - Pl

## Using SPARQL
SPARQL is more complex, but more powerful.

In [5]:
from rdflib import Graph
from rdflib.plugins.stores.sparqlstore import SPARQLStore

# Connect to NKOD SPARQL endpoint
endpoint = "https://data.gov.cz/sparql"
store = SPARQLStore(endpoint)
g = Graph(store)

# Define your query
query = """
PREFIX ds: <https://data.gov.cz/zdroj/datové-sady/00025593/>

SELECT DISTINCT ?property ?value
WHERE {
    ds:719f7b9f2cf4ab7fa40c7e7c459995a7 ?property ?value .
}
LIMIT 100
"""

for prop, val in g.query(query):
    print(f"{prop} -> {val}")

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#duration, Converter=<function parse_xsd_duration at 0x7f52d40fa160>
Traceback (most recent call last):
  File "/home/lucka/prog/EnhancedDatasetSearch/venv/lib/python3.12/site-packages/rdflib/term.py", line 2163, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
           ^^^^^^^^^^^^^^^^^^
  File "/home/lucka/prog/EnhancedDatasetSearch/venv/lib/python3.12/site-packages/rdflib/xsd_datetime.py", line 433, in parse_xsd_duration
    raise ValueError("Unable to parse duration string " + dur_string)
ValueError: Unable to parse duration string 0


http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> https://data.gov.cz/slovník/nkod/typ-datové-sady-dle-zdroje/DokumentyDCAT
http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> https://data.gov.cz/slovník/nkod/typ-datové-sady-dle-zdroje/LKOD
http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://www.w3.org/ns/dcat#Dataset
http://purl.org/dc/terms/conformsTo -> https://opendata.gov.cz/standardy:technicke-standardy-pro-datove-sady-na-stupni-3
http://purl.org/dc/terms/description -> Datová sada obsahuje statistické údaje o počtech cizinců (bez azylantů, s pobytem přechodným, dlouhodobým, pobytem trvalým a dlouhodobým vízem) v České republice, a v členění podle pohlaví, pětiletých věkových skupin a státního občanství cizince. Údaje jsou k dispozici za okresy České republiky. Připraveny jsou rovněž úhrny za Českou republiku, všechna státní občanství, pohlaví i věkové skupiny.
http://www.w3.org/ns/dqv#hasQualityMeasurement -> https://data.gov.cz/zdroj/datová-kvalita/pozorování/metrikaDostu

In [6]:
from rdflib import Graph
from rdflib.plugins.stores.sparqlstore import SPARQLStore

endpoint = "https://data.gov.cz/sparql"
g = Graph(SPARQLStore(endpoint))

query = """
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX ds: <https://data.gov.cz/zdroj/datové-sady/00025593/>
PREFIX lsgov: <https://slovník.gov.cz/legislativní/sbírka/111/2009/pojem/>

SELECT ?poskytovatel ?název ?identifikátorDatovéSchránky
WHERE {
  ds:719f7b9f2cf4ab7fa40c7e7c459995a7 dct:publisher ?poskytovatel .

  ?poskytovatel lsgov:má-název-orgánu-veřejné-moci ?název .
  ?poskytovatel lsgov:má-datovou-schránku-orgánu-veřejné-moci ?datováSchránka .

  ?datováSchránka lsgov:má-identifikátor-datové-schránky ?identifikátorDatovéSchránky .
}
"""

for owner, title, id in g.query(query):
    print("Poskytovatel:", str(owner))
    print("Název:", str(title))
    print("Identifikátor datové sady:", str(id))
    print("-" * 60)

Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/00025593
Název: Český statistický úřad
Identifikátor datové sady: 2gfaasy
------------------------------------------------------------


In [7]:
query = """
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX ds: <https://data.gov.cz/zdroj/datové-sady/00025593/>
PREFIX lsgov: <https://slovník.gov.cz/legislativní/sbírka/111/2009/pojem/>

SELECT ?poskytovatel ?nazev
WHERE {
  ?poskytovatel lsgov:má-název-orgánu-veřejné-moci ?nazev .
}
LIMIT 10
"""

for owner, title, in g.query(query):
    print("Poskytovatel:", str(owner))
    print("Název:", str(title))
    print("-" * 60)

Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/47608676
Název: Bezpečnostní informační služba
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/08884714
Název: Úřad evropského veřejného žalobce
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/70888337
Název: Jihomoravský kraj
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/70890650
Název: Jihočeský kraj
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/70891168
Název: Karlovarský kraj
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/70890749
Název: Kraj Vysočina
--